# Expert ID Comparison Notebook

This notebook helps you compare `expert_id` presence across multiple files (CSV/TSV or pandas DataFrames). It produces a summary table showing which files each `expert_id` appears in, how many files, and a pairwise overlap matrix.

Usage:
1. Provide a dictionary `files = {'fileA': 'path/to/a.csv', 'fileB': 'path/to/b.csv', ...}` **or**
   `files = {'fileA': df_a, 'fileB': df_b, ...}` where values are pandas DataFrames.
2. Run the `compare_expert_ids(files, id_col='expert_id')` function.

The default column name is `expert_id`. Change with the `id_col` argument if needed.

In [16]:
import pandas as pd
from pathlib import Path

def _read_if_path(obj):
    import pandas as pd
    from pathlib import Path
    
    if isinstance(obj, (str, Path)):
        path = str(obj)
        
        if path.lower().endswith('.csv'):
            return pd.read_csv(path)
        
        elif path.lower().endswith(('.tsv', '.txt')):
            return pd.read_csv(path, sep='\t')
        
        elif path.lower().endswith('.parquet'):
            return pd.read_parquet(path)
        
        elif path.lower().endswith(('.xlsx', '.xls')):
            return pd.read_excel(path)  # <-- Added Excel support
        
        else:
            raise ValueError(f"Unsupported file type: {path}")
    
    elif isinstance(obj, pd.DataFrame):
        return obj.copy()
    
    else:
        raise ValueError(f"Unsupported file object type: {type(obj)}")

def compare_expert_ids(files_dict, id_col='expert_id'):
    # Compare expert_id occurrences across multiple files.
    dfs = {}
    for name, val in files_dict.items():
        df = _read_if_path(val)
        if id_col not in df.columns:
            raise KeyError(f"Column '{id_col}' not found in file '{name}'. Available columns: {list(df.columns)}")
        ids = pd.Series(df[id_col].dropna().astype(str).unique(), name=id_col)
        dfs[name] = pd.DataFrame(ids)

    all_ids = sorted({i for df in dfs.values() for i in df[id_col].tolist()})
    presence = pd.DataFrame(False, index=all_ids, columns=list(dfs.keys()))
    for name, df in dfs.items():
        presence.loc[df[id_col].astype(str), name] = True

    summary = presence.copy()
    summary['total_files'] = summary.sum(axis=1)
    summary['files_list'] = summary.apply(lambda row: [col for col in dfs.keys() if row[col]], axis=1)
    summary_df = summary[['total_files', 'files_list']].reset_index().rename(columns={'index': id_col})
    summary_df = summary_df.sort_values(['total_files', id_col], ascending=[False, True]).reset_index(drop=True)

    file_names = list(dfs.keys())
    pairwise = pd.DataFrame(0, index=file_names, columns=file_names)
    for i in range(len(file_names)):
        for j in range(i, len(file_names)):
            a = set(dfs[file_names[i]][id_col].astype(str))
            b = set(dfs[file_names[j]][id_col].astype(str))
            cnt = len(a & b)
            pairwise.iat[i, j] = cnt
            pairwise.iat[j, i] = cnt

    return {'summary_df': summary_df, 'pairwise': pairwise, 'presence_matrix': presence}


In [17]:
import pandas as pd
from IPython.display import display

df_a = pd.DataFrame({'expert_id': ['E1','E2','E3','E5'], 'value':[10,20,30,50]})
df_b = pd.DataFrame({'expert_id': ['E2','E3','E4'], 'value':[7,8,9]})
df_c = pd.DataFrame({'expert_id': ['E1','E4','E6'], 'value':[1,2,3]})

files = {'file_A': df_a, 'file_B': df_b, 'file_C': df_c}

results = compare_expert_ids(files, id_col='expert_id')

print('Summary (expert_id, total_files, files_list):')
display(results['summary_df'])

print('Pairwise overlap counts (files x files):')
display(results['pairwise'])

print('Presence matrix (1 = present):')
display(results['presence_matrix'].astype(int))


Summary (expert_id, total_files, files_list):


,expert_id,total_files,files_list
0,E1,2,"[file_A, file_C]"
1,E2,2,"[file_A, file_B]"
2,E3,2,"[file_A, file_B]"
3,E4,2,"[file_B, file_C]"
4,E5,1,[file_A]
5,E6,1,[file_C]


Pairwise overlap counts (files x files):


,file_A,file_B,file_C
file_A,4,2,1
file_B,2,3,1
file_C,1,1,3


Presence matrix (1 = present):


,file_A,file_B,file_C
E1,1,0,1
E2,1,1,0
E3,1,1,0
E4,0,1,1
E5,1,0,0
E6,0,0,1


In [18]:
from pathlib import Path

root = Path(r"C:\Users\nathan.taylor\Asurion\Coaching & Technology - BPC Data\Locked_Cohorts")

files = {
    "VZW_SOL_CRT_Hangup_cohort_2_2026-02-12": root / "VZW_SOL_CRT_Hangup_cohort_2_2026-02-12.xlsx",
    "VZW_SOL_CRT_Callback_cohort_2_2026-02-12": root / "VZW_SOL_CRT_Callback_cohort_2_2026-02-12.xlsx",
    "VZW_SOL_CRT_Holdtime_cohort_2_2026-02-12": root / "VZW_SOL_CRT_Holdtime_cohort_2_2026-02-12.xlsx",
}

In [19]:
def write_results(results, out_prefix='expert_compare'):
    results['summary_df'].to_csv(f"{out_prefix}_summary.csv", index=False)
    results['pairwise'].to_csv(f"{out_prefix}_pairwise.csv")
    results['presence_matrix'].to_csv(f"{out_prefix}_presence_matrix.csv")
    print('Wrote files:', f"{out_prefix}_summary.csv", f"{out_prefix}_pairwise.csv", f"{out_prefix}_presence_matrix.csv")


In [20]:
results = compare_expert_ids(files, id_col="expert_id")

results["summary_df"]

,expert_id,total_files,files_list
0,528718,3,"[VZW_SOL_CRT_Hangup_cohort_2_2026-02-12, VZW_S..."
1,541081,3,"[VZW_SOL_CRT_Hangup_cohort_2_2026-02-12, VZW_S..."
2,546780,3,"[VZW_SOL_CRT_Hangup_cohort_2_2026-02-12, VZW_S..."
3,589541,3,"[VZW_SOL_CRT_Hangup_cohort_2_2026-02-12, VZW_S..."
4,594316,3,"[VZW_SOL_CRT_Hangup_cohort_2_2026-02-12, VZW_S..."
...,...,...,...
1187,703083,1,[VZW_SOL_CRT_Hangup_cohort_2_2026-02-12]
1188,703091,1,[VZW_SOL_CRT_Hangup_cohort_2_2026-02-12]
1189,703211,1,[VZW_SOL_CRT_Hangup_cohort_2_2026-02-12]
1190,703561,1,[VZW_SOL_CRT_Hangup_cohort_2_2026-02-12]


In [21]:
output_dir = r"C:\Users\nathan.taylor\Asurion\Coaching & Technology - BPC Data\Proposed_Cohorts"

results["summary_df"].to_csv(f"{output_dir}\\locked_cohort_compare_summary.csv", index=False)
results["pairwise"].to_csv(f"{output_dir}\\locked_cohort_compare_pairwise.csv")
results["presence_matrix"].to_csv(f"{output_dir}\\locked_cohort_compare_presence_matrix.csv")

## Using real files on disk

If you have CSV/TSV files on disk, provide a dictionary like:

```python
files = {
  'jan': '/path/to/january.csv',
  'feb': '/path/to/feb.csv'
}
results = compare_expert_ids(files, id_col='expert_id')
```

Then inspect `results['summary_df']` etc.